In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"


In [0]:
tables = spark.sql(f"SHOW TABLES IN {silver_schema}")
display(tables)

##gold.dim_motivo

In [0]:
#gold.dim_motivo (origem silver.tb_motivo)

# partição única explícita: a dimensão é pequena, então juntar tudo numa partição não pesa
w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_motivo").cast("int"))

df_silver_motivo = spark.read.table(f"{silver_schema}.tb_base_motivos")

df_dim_motivo = (
    df_silver_motivo
    .withColumn("SK_motivo", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_motivo"),
        F.col("id_motivo").cast("int").alias("id_motivo"),
        F.col("nome_motivo").cast("string").alias("nome"),
        F.col("categoria").cast("string").alias("categoria"),
        F.col("criticidade").cast("string").alias("criticidade"),
    )
    .orderBy("SK_motivo")
)

display(df_dim_motivo)

df_dim_motivo.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_motivo")

## gold.dim_canal

In [0]:
#gold.dim_canal (origem silver.tb_canais)

w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_canal").cast("int"))

df_silver_canais = spark.read.table(f"{silver_schema}.tb_canais")

# a silver não traz etapa_funil nem nivel_etapa: derivamos dos valores padronizados na Silver
k_nome_canal = F.lower(F.trim(F.col("nome_canal")))

etapa_funil = (
    F.when(k_nome_canal == "ura", F.lit("Autosserviço"))
    .when(k_nome_canal == "chatbot", F.lit("Autosserviço"))
    .when(k_nome_canal == "atendimento inicial", F.lit("Atendimento Inicial"))
    .when(k_nome_canal == "atendimento especializado", F.lit("Atendimento Especializado"))
    .otherwise(F.lit("Desconhecido"))
)

# ordem manual, para refletir a posição da etapa no funil de atendimento
nivel_etapa = (
    F.when(etapa_funil == "Autosserviço", F.lit(1))
    .when(etapa_funil == "Atendimento Inicial", F.lit(2))
    .when(etapa_funil == "Atendimento Especializado", F.lit(3))
    .otherwise(F.lit(None).cast("int"))
)

df_dim_canal = (
    df_silver_canais
    .withColumn("SK_canal", F.row_number().over(w_sk).cast("bigint"))
    .withColumn("etapa_funil", etapa_funil)
    .withColumn("nivel_etapa", nivel_etapa)
    .select(
        F.col("SK_canal"),
        F.col("id_canal").cast("int").alias("id_canal"),
        F.col("nome_canal").cast("string").alias("nome"),
        F.col("tipo_canal").cast("string").alias("tipo"),
        F.col("etapa_funil").cast("string").alias("etapa_funil"),
        F.col("nivel_etapa").cast("int").alias("nivel_etapa"),
        F.col("status_canal").cast("string").alias("status"),
    )
    .orderBy("SK_canal")
)

display(df_dim_canal)

df_dim_canal.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_canal")

## gold.dim_atendente

In [0]:
#gold.dim_atendente (origem silver.tb_base_atendentes)

w_sk = Window.partitionBy(F.lit(1)).orderBy(F.col("id_atendente").cast("int"))

df_silver_atendentes = spark.read.table(f"{silver_schema}.tb_base_atendentes")

df_dim_atendente = (
    df_silver_atendentes
    .withColumn("SK_atendente", F.row_number().over(w_sk).cast("bigint"))
    .select(
        F.col("SK_atendente"),
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.col("nome_atendente").cast("string").alias("nome"),
        F.col("nivel_atendimento").cast("int").alias("nivel_atendimento"),
        F.col("data_admissao").cast("date").alias("data_admissao"),
        F.col("data_desligamento").cast("date").alias("data_desligamento"),
        F.col("status_atendente").cast("string").alias("status"),
    )
    .orderBy("SK_atendente")
)

display(df_dim_atendente)

df_dim_atendente.write.mode("overwrite").saveAsTable(f"{gold_schema}.dim_atendente")